# DATA 5310 HW 8: Group Project EDA

**Group:** Maya Silver, Gurpreet Kaur, Badamgarav Battushig

This notebook combines the three datasets our group is using for the final project:

1. **EuRepoC** Global Dataset of Cyber Incidents (Maya Silver): politically significant cyber incidents, 2001 to 2024.
2. **Ransomware.live** victim claims (Gurpreet Kaur): ransomware leak-site victims, December 2021 to November 2026.
3. **World Bank** country indicators (Badamgarav Battushig): population, GDP per capita and internet use by country and year.

The notebook is in two parts. **Data** introduces, describes and cleans each dataset. **EDA** explores each one. Run the cells in order from top to bottom, because later cells use variables from earlier ones.

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()


## Data

### Dataset 1: EuRepoC Global Dataset of Cyber Incidents (Maya Silver)

#### Data description

My dataset is the Global Dataset of Cyber Incidents (version 1.3) from the European Repository of Cyber Incidents (EuRepoC), downloaded from Zenodo. EuRepoC is a research project that tracks publicly reported cyber incidents with a political dimension, such as attacks by state-backed groups or attacks on governments and critical infrastructure. Despite the name, it covers incidents worldwide, with start dates from 2000 through 2024.

The file has 3,414 rows and 85 columns, and one row is one cyber incident. The columns fall into a few groups: basic details (ID, name, description, start and end dates), the incident type (data theft, disruption, hijacking, ransomware), the target (receiver name, country, region, and sector), the attacker (initiator name, country, and category, such as state or non-state), severity and impact scores (weighted intensity, impact indicator, economic impact), political and legal responses, and technical details like zero-day use and MITRE ATT&CK initial access and impact techniques.

Source: European Repository of Cyber Incidents (EuRepoC), Global Dataset of Cyber Incidents, version 1.3, Zenodo, March 2025. https://zenodo.org/records/14965395

There are a few challenges to be aware of:

- Many cells hold several values separated by semicolons. For example, an incident can have two incident types or several target countries, so these have to be split before counting.
- Missing data is mostly stored as text like "Not available", "Unknown", or "Not attributed" instead of true nulls. That's why info() shows almost no missing values even though a lot is missing in practice.
- Dates are stored as text in day.month.year format and need to be converted.
- The data only includes publicly reported incidents, so it reflects what gets reported and attributed, not everything that happens. Attribution is also uncertain. For incidents targeting the US, the two biggest attacker categories are "Not attributed" and "Unknown".

In [ ]:
# Data Set

eurepoc_raw = pd.read_csv('https://raw.githubusercontent.com/SU-DataViz-Group-Fall-2026/ransomware-analysis/main/data/eurepoc_global_dataset_1_3.csv')
eurepoc_raw.head()

In [ ]:
eurepoc_raw.columns

In [ ]:
eurepoc_raw.shape

#### Data cleaning

I kept 45 of the 85 columns, focusing on the incident type, target, attacker, dates, severity, impact, political response, and MITRE fields. For the exploration, I filtered to incidents where the United States is one of the target countries, which leaves 871 incidents. I converted the start date to a date type and the intensity and impact scores to numbers. I also wrote a small helper that splits the multi-value cells and counts each value separately, so an incident with two types is counted once for each type.


In [ ]:
cyber = eurepoc_raw[['incident_id', 'name', 'description',
            'incident_type', 'inclusion_criterion', 'source_disclosure',
            'receiver_name', 'receiver_country', 'receiver_regions',
            'receiver_category', 'receiver_subcategory',
            'initiator_name', 'initiator_country',
            'initiator_category', 'initiator_subcategory', 'settled_initiator',
            'start_date', 'end_date', 'attribution_date', 'political_response_date',
            'unweighted_intensity', 'target_multiplier', 'weighted_intensity',
            'impact_indicator_label', 'impact_indicator_score',
            'economic_impact', 'economic_impact_value', 'economic_impact_currency',
            'affected_entities', 'intelligence_impact', 'functional_impact',
            'number_political_responses', 'political_response_type',
            'political_response_responding_country', 'political_response_responding_actor',
            'zero_days', 'has_disruption', 'data_theft', 'disruption', 'hijacking',
            'physical_effects_spatial', 'physical_effects_temporal', 'user_interaction',
            'mitre_initial_access', 'mitre_impact']]

cyber.head()

In [ ]:
cyber.info()

### Dataset 2: Ransomware.live Victim Claims (Gurpreet Kaur)

#### Load data
Ransomware.live has no single download file. Its public API returns the victims for one month at a time (`/victims/<year>/<month>`). We downloaded every month from 2021 to the download date once, pausing between requests because the API rate-limits, and saved the result as a fixed snapshot in `data/ransomware_live_raw.json`. This notebook reads that snapshot and does not call the API, so the numbers stay the same each time it runs.

In [ ]:
import json
import urllib.request
import time
from pathlib import Path

%pip install pycountry

import altair as alt
import pandas as pd
import pycountry
import requests

alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [ ]:
RAW_URL = "https://raw.githubusercontent.com/SU-DataViz-Group-Fall-2026/ransomware-analysis/main/data/ransomware_live_raw.json"

with urllib.request.urlopen(RAW_URL, timeout=120) as resp:
    rows = json.loads(resp.read().decode("utf-8"))
print(f"Loaded snapshot: {len(rows):,} records from {RAW_URL}")

rw_raw = pd.DataFrame(rows)


#### Describe

In [ ]:
rw_raw.head()

In [ ]:
print(f"Rows: {len(rw_raw):,}    Columns: {rw_raw.shape[1]}")
print("Columns:", list(rw_raw.columns))

nested = [c for c in rw_raw.columns if rw_raw[c].map(lambda v: isinstance(v, (dict, list))).any()]
print("\nNested columns:", nested)

rw_raw.info()

#### Describing the data

The data comes from Ransomware.live, a project by security researcher Julien Mousqueton that monitors the leak sites where ransomware groups name the organizations they have attacked. We downloaded it from the free v2 API on Nov 16, 2026, one month at a time.

Each row is one claim, a victim organization listed by a ransomware group on its leak site. The same organization can appear more than once. As an example, claimed by two groups or re-posted.

For the columns, the core fields are organization name which are victims, group (the ransomware group that claimed it), attackdate is date of the incident and country (ISO-2 code, such as US or DE). The API also returns nested fields that we leave out of the analysis table. Other columns in our download. There are 15 rows and 3 columns before cleaning.

#### Challenges
1. A leak site only shows victims that the group chose to name. Victims who paid quietly, and groups that do not run leak sites, are invisible. Counts are a lower bound, not all ransomware attacks. 
2. Ransomware.live keeps adding groups and sites to its monitoring, so part of any upward trend may reflect better tracking rather than more attacks.
3. Attackdate may be an estimate or the posting date, and the month endpoint may be organized by when a victim was discovered. We test this below.
4. Country and sector are not always filled in.
5. The same victim can appear several times.
6. The numbers change daily, so we cache a snapshot and cite the access date.
7. Victim names are real organizations, many of them hospitals, schools or small businesses. We aggregate before plotting and do not put names in public charts.
8. The site has a Terms & Conditions page. We cite the source and use the data for coursework only.

In [ ]:
rw = rw_raw.drop(columns=nested).copy()
rw.columns = rw.columns.str.strip().str.lower()

placeholders = {"", "not found", "unknown", "n/a", "none"}
for c in rw.select_dtypes(include=["object", "string"]).columns:
    s = rw[c].astype("string").str.strip()
    rw[c] = s.mask(s.str.lower().isin(placeholders))

print("Share of missing values per column (%):")
print((rw.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

In [ ]:
def to_date(series):
    '''Parse mixed date strings to timezone-free datetimes; unparseable values become NaT.'''
    try:
        parsed = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    except (TypeError, ValueError):                
        parsed = pd.to_datetime(series, errors="coerce", utc=True)
    return parsed.dt.tz_localize(None)


rw["attack_date"] = to_date(rw["attackdate"])
if "discovered" in rw.columns:
    rw["discovered_date"] = to_date(rw["discovered"])

n0 = len(rw)

dupe_key = [c for c in ["victim", "group", "attack_date"] if c in rw.columns]
rw = rw.drop_duplicates(subset=dupe_key)
print(f"Duplicate rows removed:      {n0 - len(rw):,}")

ok = rw["attack_date"].between(pd.Timestamp("2013-01-01"), pd.Timestamp.today())
print(f"Bad or missing dates removed: {(~ok).sum():,}")
rw = rw[ok].copy()

rw["year"] = rw["attack_date"].dt.year
rw["year_month"] = rw["attack_date"].dt.to_period("M").dt.to_timestamp()
print(f"Rows kept: {len(rw):,}  (from {n0:,})")
print(f"Attack dates run from {rw['attack_date'].min():%Y-%m-%d} to {rw['attack_date'].max():%Y-%m-%d}")

In [ ]:
iso2_to_name = {c.alpha_2: c.name for c in pycountry.countries}
rw["country_code"] = rw["country"].str.upper().replace({"UK": "GB"})
unknown_codes = sorted(set(rw.loc[rw["country_code"].notna() & ~rw["country_code"].isin(iso2_to_name), "country_code"]))
print("Codes that are not valid ISO-2 (set to missing):", unknown_codes)
rw.loc[~rw["country_code"].isin(iso2_to_name), "country_code"] = pd.NA
rw["country_name"] = rw["country_code"].map(iso2_to_name)

sector_col = next((c for c in ["activity", "sector"] if c in rw.columns), None)
if sector_col:
    rw["sector"] = rw[sector_col]
    print(f"Sector column found: '{sector_col}' ({rw['sector'].nunique()} distinct values)")
else:
    print("No sector column in this download. Check rw_raw.columns or /sectorvictims/<sector>.")

print(f"\nRows with a country: {rw['country_code'].notna().mean():.1%}")
if sector_col:
    print(f"Rows with a sector:  {rw['sector'].notna().mean():.1%}")

In the above steps I have done tidy column names and placeholder text, parse dates, dropped duplicates and impossible dates, convert country codes to names, and find the sector column.

In [ ]:
if "requested_month" in rw.columns:
    share = (rw["attack_date"].dt.strftime("%Y-%m") == rw["requested_month"]).mean()
    print(f"{share:.1%} of records have an attack date in the month they were requested under")

if "discovered_date" in rw.columns:
    lag = (rw["discovered_date"] - rw["attack_date"]).dt.days
    print("\nDays between attack date and discovery date:")
    print(lag.describe().round(1))

The month endpoint follows attackdate closely: 97.4% of records have an attack date in the month they were requested under, so grouping by attackdate is safe for our monthly and yearly counts. The gap between the attack date and the discovery date is short for most victims. Across the 31,081 records that have both dates, the median gap is 0 days and 75% of records are within 7 days. The average (19.7 days) is much higher than the median because of a few extreme values (standard deviation 78.7 days), with the largest gap being 1,211 days, a victim posted more than three years after the attack. The minimum is -1,144 days, which means a victim was "discovered" about three years before its attack date. This is impossible and points to data errors or back-dated entries. Because at least a quarter of records have a gap of exactly 0 days, attackdate is often probably the same as the date the victim appeared on the leak site rather than the true date of the attack. We therefore keep attackdate as our time variable but treat our monthly trends as approximate, showing when victims were publicly named.

### Dataset 3: World Bank Country Indicators (Badamgarav Battushig)

#### About this dataset

Source: World Bank (2026). *World Development Indicators*. Retrieved November 16, 2026, from https://data.worldbank.org

The World Bank publishes yearly statistics for almost every country in the world. We use three of them:

| Indicator | Code | Source page |
|---|---|---|
| Population, total | `SP.POP.TOTL` | https://data.worldbank.org/indicator/SP.POP.TOTL |
| GDP per capita (current US$) | `NY.GDP.PCAP.CD` | https://data.worldbank.org/indicator/NY.GDP.PCAP.CD |
| Individuals using the Internet (% of population) | `IT.NET.USER.ZS` | https://data.worldbank.org/indicator/IT.NET.USER.ZS |

The files were downloaded from the pages above and saved in  my GitHub repository: https://github.com/SU-DataViz-Group-Fall-2026/ransomware-analysis/tree/main/data

Data Choice: A large country like the United States will naturally have more attacks than a small one, simply because it has more people, companies, and computers. This dataset tells us how many people live in each country, how wealthy it is, and how many people use the internet. With it we can make fair comparisons, such as **attacks per million people**, and check whether richer or more connected countries are targeted more often.

In [ ]:
# install country_converter (only needed once)
%pip install -q country_converter

In [ ]:
import pandas as pd
import altair as alt
import country_converter as coco

#### Data Set

In [ ]:
# GitHub location of our data files
github_url = 'https://raw.githubusercontent.com/SU-DataViz-Group-Fall-2026/ransomware-analysis/main/data/'

# the 3 indicator files start with 4 rows of notes, so we skip them
population = pd.read_csv(github_url + 'wb_population.csv', skiprows=4)
gdp = pd.read_csv(github_url + 'wb_gdp_per_capita.csv', skiprows=4)
internet = pd.read_csv(github_url + 'wb_internet_users.csv', skiprows=4)

# country metadata (region and income group for each country)
metadata = pd.read_csv(github_url + 'wb_country_metadata.csv')

population.head()

In [ ]:
population.columns

In [ ]:
population.shape

In [ ]:
metadata.head()

In [ ]:
metadata.shape

#### Description of the data

Loaded four csv files.

Indicator files (population, GDP per capita, internet users)
- Each row is one country or region. Every year from 1960 to 2025 has its own column, so you read across a row to see how a country changed over time.
- Each file has 266 rows and 71 columns.
- Columns: `Country Name`, `Country Code` (a 3-letter code, e.g. USA), `Indicator Name`, `Indicator Code`, one column per year, and an empty column at the end caused by an extra comma in the file.

Metadata file
- Each row is one country or region, with the world region it belongs to (e.g. North America) and its income group (low, lower middle, upper middle, or high income).
- It has 265 rows.

Things to know before using the data

1. Extra notes at the top. Each file has 4 lines of notes before the real table starts. We skip those lines.

2. Years are sideways. Each year has its own column (2000, 2001, 2002...). That makes charts hard, so we turn the table so all the years go in one "year" column.

3. Some rows are not countries. 48 rows are groups like "World" or "High income." If we keep them, they get counted like real countries. We can find them easily because they have no region listed, so remove them.

4. Some numbers are missing. 2025 is mostly empty, so only use 2000–2024. North Korea is one of the main attacker countries, but it has no GDP numbers and only 5 years of internet numbers.

5. Country names do not match. The World Bank writes countries one way (USA, Russian Federation). EuRepoC and Ransomware.live write them another way (US). We add the short 2-letter code to the World Bank data so all the files can be joined.

#### Data Cleaning

In [ ]:
# helper: turn one indicator file from wide (one column per year) to long (one year column)
def to_long(data, value_name):
    long = data.melt(id_vars=['Country Name', 'Country Code'],
                     value_vars=[str(y) for y in range(2000, 2025)],
                     var_name='year', value_name=value_name)
    return long

# keep 2000-2024 (2025 is mostly empty) and melt each file
pop_long = to_long(population, 'population')
gdp_long = to_long(gdp, 'gdp_per_capita')
net_long = to_long(internet, 'internet_users_pct')

pop_long.head()

In [ ]:
# combine the 3 indicators into one table: one row per country per year
wb = pop_long.merge(gdp_long, on=['Country Name', 'Country Code', 'year'])
wb = wb.merge(net_long, on=['Country Name', 'Country Code', 'year'])

# rename columns so they are easier to read
wb = wb.rename(columns={'Country Name': 'country', 'Country Code': 'iso3'})

wb.head()

In [ ]:
# keep only the columns we need from the metadata and rename them
meta = metadata[['Country Code', 'Region', 'IncomeGroup']]
meta = meta.rename(columns={'Country Code': 'iso3', 'Region': 'region', 'IncomeGroup': 'income_level'})

# remove regions like "World" and "High income" (they have no Region in the metadata)
real_countries = meta.dropna(subset=['region'])
print(len(meta), 'rows in metadata,', len(real_countries), 'real countries')

# keep only real countries and add region and income level
wb = wb.merge(real_countries, on='iso3', how='inner')
print(wb.shape)

In [ ]:
# rename income levels to shorter labels for charts
wb['income_level'] = wb['income_level'].replace({'Low income': 'Low',
                                                 'Lower middle income': 'Lower middle',
                                                 'Upper middle income': 'Upper middle',
                                                 'High income': 'High'})

# shorten long country names so they read better in charts
wb['country'] = wb['country'].replace({'Russian Federation': 'Russia',
                                       'Iran, Islamic Rep.': 'Iran',
                                       "Korea, Dem. People's Rep.": 'North Korea',
                                       'Korea, Rep.': 'South Korea',
                                       'Egypt, Arab Rep.': 'Egypt',
                                       'Turkiye': 'Turkey',
                                       'Viet Nam': 'Vietnam'})

# make year a whole number
wb['year'] = wb['year'].astype(int)

In [ ]:
# check missing values
wb.isna().sum()

In [ ]:
# remove rows with no population (we can't compute "per million people" without it)
# we keep rows with missing GDP or internet use, so we don't lose the population data
wb = wb.dropna(subset=['population'])

# population in millions is easier to read and needed for "attacks per million people"
wb['population_millions'] = wb['population'] / 1000000

In [ ]:
# add a 2-letter country code so we can join with EuRepoC and Ransomware.live
# (Channel Islands has no 2-letter code, so it shows as "not found")
codes = list(wb['iso3'].unique())
iso2_codes = coco.convert(codes, to='ISO2')
wb['iso2'] = wb['iso3'].map(dict(zip(codes, iso2_codes)))

print(wb.shape)
wb.head()

**After cleaning, each row is one country in one year.**

## EDA

### Dataset 1: EuRepoC, US-targeted incidents (Maya Silver)

#### Summary and next steps

A few patterns stood out in the US incidents:

- Hijacking with misuse is the most common incident type (about 530), followed by data theft (about 430), disruption (about 320), and ransomware (about 210).
- Reported incidents stay low through 2020, with roughly 20 to 45 per year from 2011 on, then jump to about 70 in 2022, about 235 in 2023, and about 180 in 2024. Some of this is probably real growth, but some may be from better tracking and reporting in recent years, so it needs a closer look.
- Among incidents with a known attacker country, China is first (about 135), then Russia (about 95), Iran (about 60), and North Korea (about 30). But most incidents have no attributed country at all.
- Critical infrastructure and state institutions are the most targeted sectors, followed by corporate targets.
- Most incidents are low severity. Weighted intensity is mostly between 1 and 5 on a scale that goes up to 9 here, with peaks at 1 and 4, and only a handful above 6. Most incidents with an impact rating are rated Low, and many have no rating.

Patterns worth investigating next: whether state-backed attackers cause more severe incidents than non-state ones, which attacker countries target which sectors, how severity has changed over time, and which MITRE initial access techniques are most common for each incident type.

This dataset would be stronger combined with others. MITRE ATT&CK data could add detail on the named groups and their techniques (joined on group name), and the CISA Known Exploited Vulnerabilities catalog could show whether spikes in incidents line up with newly exploited vulnerabilities. Group names will be the main challenge for merging, since the same group goes by several names.

Visualizations I'm interested in making: a line chart of incidents over time by attacker country, a heatmap of attacker country by target sector, a world map of target countries, a comparison of severity by attacker type, and a ranked bar chart of MITRE techniques.


In [ ]:
us = cyber[cyber['receiver_country'].str.contains('United States', na=False)].copy()
us['start_date'] = pd.to_datetime(us['start_date'], errors='coerce')
print(len(us), 'incidents with the US as a target')

# helper: split cells with several values (a; b; c) and count each value
def count_values(data, col):
    out = (data[col].str.split(';').explode().str.strip()
           .value_counts().reset_index())
    out.columns = [col, 'incidents']
    return out

In [ ]:
types = count_values(us, 'incident_type')

alt.Chart(types).mark_bar().encode(
    alt.X('incidents:Q', title='Number of incidents'),
    alt.Y('incident_type:N', sort='-x', title='Incident type')
).properties(title='Types of Cyber Incidents Targeting the United States', width=450)

In [ ]:
by_year = us.groupby(us['start_date'].dt.year).size().reset_index(name='incidents')
by_year.columns = ['year', 'incidents']

alt.Chart(by_year).mark_bar().encode(
    alt.X('year:O', title='Start year'),
    alt.Y('incidents:Q', title='Number of incidents')
).properties(title='Cyber Incidents Targeting the United States by Year', width=500)

In [ ]:
attackers = count_values(us, 'initiator_country').head(10)

alt.Chart(attackers).mark_bar().encode(
    alt.X('incidents:Q', title='Number of incidents'),
    alt.Y('initiator_country:N', sort='-x', title='Attacker country')
).properties(title='Top Attacker Countries for Incidents Targeting the United States', width=450)

In [ ]:
sectors = (us['receiver_category'].str.split(';')
           .apply(lambda x: list(set(s.strip() for s in x)))
           .explode().value_counts().reset_index())
sectors.columns = ['receiver_category', 'incidents']

alt.Chart(sectors).mark_bar().encode(
    alt.X('incidents:Q', title='Number of incidents'),
    alt.Y('receiver_category:N', sort='-x', title='Target sector')
).properties(title='US Sectors Targeted in Cyber Incidents', width=450)

In [ ]:
us['weighted_intensity'] = pd.to_numeric(us['weighted_intensity'], errors='coerce')
us['impact_indicator_score'] = pd.to_numeric(us['impact_indicator_score'], errors='coerce')
us['year'] = us['start_date'].dt.year

In [ ]:
alt.Chart(us).mark_bar().encode(
    alt.X('weighted_intensity:Q', bin=alt.Bin(step=1), title='Weighted cyber intensity'),
    alt.Y('count():Q', title='Number of incidents')
).properties(title='How Severe Are Cyber Incidents Targeting the US?', width=450)

In [ ]:
alt.Chart(us).mark_circle(opacity=0.4, size=50).encode(
    alt.X('start_date:T', title='Start date'),
    alt.Y('weighted_intensity:Q', title='Weighted cyber intensity'),
    tooltip=['name:N', 'start_date:T', 'weighted_intensity:Q', 'incident_type:N']
).properties(title='Severity of Incidents Targeting the US Over Time', width=600)

In [ ]:
by_type = us.assign(incident_type=us['incident_type'].str.split(';')).explode('incident_type')
by_type['incident_type'] = by_type['incident_type'].str.strip()

alt.Chart(by_type).mark_rect().encode(
    alt.X('impact_indicator_label:N', title='Impact level', sort=['Minor', 'Low', 'Medium', 'Not available']),
    alt.Y('incident_type:N', title='Incident type'),
    alt.Color('count():Q', title='Incidents', scale=alt.Scale(scheme='blues')),
    tooltip=['incident_type:N', 'impact_indicator_label:N', 'count():Q']
).properties(title='Which Incident Types Have the Biggest Impact?', width=350, height=200)

### Dataset 2: Ransomware.live victims (Gurpreet Kaur)

#### Exploratory charts of the Ransomware.live data
Three charts of the cleaned table rw, one for each question we can later compare with EuRepoC: when victims were posted, where they are, and which sectors they belong to. The same three questions are the ones the two datasets share (dates, countries and sectors or target categories).

Two choices apply to all three charts:
- The API returned no victims before April 2022, and the current month is still in progress. We therefore chart complete months only, from April 2022 up to last month.
- All charts use the Ransomware.live red that our group uses for this dataset.

In [ ]:
RW_COLOR = "#e45756"        

first_month = pd.Timestamp("2022-04-01")                      
this_month = pd.Timestamp.today().to_period("M").to_timestamp()  
rw_eda = rw[(rw["year_month"] >= first_month) & (rw["year_month"] < this_month)].copy()

print(f"{len(rw_eda):,} of {len(rw):,} victims fall in complete months "
      f"({rw_eda['year_month'].min():%b %Y} to {rw_eda['year_month'].max():%b %Y})")

**Chart 1. When were victims posted?** 

In [ ]:
monthly = rw_eda.groupby("year_month").size().rename("victims").reset_index()
first, last = monthly["year_month"].min(), monthly["year_month"].max()

alt.Chart(monthly).mark_line(color=RW_COLOR, point=alt.OverlayMarkDef(color=RW_COLOR)).encode(
    x=alt.X("year_month:T", title="Month", axis=alt.Axis(format="%Y", tickCount="year")),
    y=alt.Y("victims:Q", title="Victims posted"),
    tooltip=[alt.Tooltip("year_month:T", title="Month", format="%B %Y"),
             alt.Tooltip("victims:Q", title="Victims", format=",")]
).properties(title=f"Ransomware Victims Posted per Month ({first:%b %Y} to {last:%b %Y})",
             width=650, height=300)

How many victims were posted each month, and is the number growing? That part of any growth can come from Ransomware.live monitoring more groups over time (challenge 2), and that attackdate is often the date the victim was posted.

**Chart 2. Where are the victims?**

In [ ]:
known = rw_eda.dropna(subset=["country_name"])
top_countries = (known["country_name"].value_counts().head(10)
                      .rename_axis("country").reset_index(name="victims"))

bars = alt.Chart(top_countries).mark_bar(color=RW_COLOR).encode(
    x=alt.X("victims:Q", title="Number of victims", axis=alt.Axis(tickCount=6),
            scale=alt.Scale(domainMax=top_countries["victims"].max() * 1.12)),
    y=alt.Y("country:N", sort="-x", title=None),
    tooltip=["country:N", alt.Tooltip("victims:Q", format=",")])
labels = bars.mark_text(align="left", dx=3).encode(text=alt.Text("victims:Q", format=","))

(bars + labels).properties(
    title=alt.TitleParams("Top 10 Countries by Ransomware Victims",
                          subtitle=f"Only the {len(known) / len(rw_eda):.0%} of victims that have a country"),
    width=500)

Only victims with a country are counted, and the title shows what share that is. These are raw counts, so large countries are expected to lead. The World Bank table (iso2, population_millions) will let us turn this into victims per million people.

**Chart 3. Which sectors are hit, and did the mix change?** 

In [ ]:
sec = rw_eda.dropna(subset=["sector"])
sec_year = sec.groupby(["year", "sector"]).size().rename("victims").reset_index()
sec_year["share"] = sec_year["victims"] / sec_year.groupby("year")["victims"].transform("sum")
sector_order = sec["sector"].value_counts().index.tolist()
dark = (sec_year["share"].min() + sec_year["share"].max()) / 2   # cells above the midpoint get white text

cells_ = alt.Chart(sec_year).mark_rect().encode(
    x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("sector:N", sort=sector_order, title=None),
    color=alt.Color("share:Q", scale=alt.Scale(scheme="reds"),
                    legend=alt.Legend(title="Share of year", format=".0%")),
    tooltip=["year:O", "sector:N", alt.Tooltip("victims:Q", format=","), alt.Tooltip("share:Q", format=".1%")])
numbers = alt.Chart(sec_year).mark_text(fontSize=10).encode(
    x="year:O", y=alt.Y("sector:N", sort=sector_order),
    text=alt.Text("share:Q", format=".0%"),
    color=alt.condition(f"datum.share > {dark}", alt.value("white"), alt.value("black")))

(cells_ + numbers).properties(
    title=alt.TitleParams("Share of Each Year's Victims by Sector",
                          subtitle=f"Only the {len(sec) / len(rw_eda):.0%} of victims that have a sector"),
    width=330, height=330)

For each year, what share of victims belongs to each sector? Shares (not counts) are used because 2022 (April to December) and 2026 (January to last month) are partial years, and the sector label is missing for a small part of the data. Sectors are sorted from most to least common overall.


#### Summary

Ransomware.live is a large, fast-growing list of criminal victims with a date, a country and a sector. It is the volume side of the story. EuRepoC is the opposite: a smaller list of *politically significant* incidents with rich detail about who attacked whom. Together they let us ask whether the two kinds of attacks follow the same geography and timeline.

**Patterns and relationships worth investigating**
1. **Trend over time.** Do monthly ransomware victims rise after 2020, and does the jump in EuRepoC incidents against the US (about 70 in 2022 to about 236 in 2023 in our teammate's chart) coincide with it? We must keep in mind that part of the growth is better monitoring (challenge 2).
2. **Geography.** Which countries have the most victims, and does that change once we divide by population? Are those the same countries that appear most often as targets in EuRepoC?
3. **Do the two datasets move together?** A country-level correlation between EuRepoC incidents and ransomware victims, and a direct comparison of EuRepoC's own *Ransomware* incidents (about 210 against the US) with the far larger number of leak-site victims. The gap shows how little of ransomware reaches the "politically significant" list.
4. **Concentration among groups.** Are a handful of groups responsible for most victims, and how quickly do top groups appear and disappear?
5. **Sectors.** Which sectors does ransomware hit compared with EuRepoC's receiver categories (critical infrastructure, state institutions, corporate targets)?

It needs more data we can use from below:
- **Population** (World Bank, added in 2d) to normalize country counts. A possible fourth source is the ITU Global Cybersecurity Index, to test whether countries with higher cybersecurity scores see fewer victims.
- A hand-made **crosswalk** between Ransomware.live's sector labels and EuRepoC's `receiver_category`, because the two use different vocabularies.

**Still to do.** Confirm the real column names after the first full download, decide how to treat the missing country and sector values, and agree with the group on the 2020-2024 window and on which EuRepoC incidents count as "state-linked" (the `initiator_category` column looks useful).

**Visualizations I am interested in:** a monthly trend with both datasets, a log-log country scatter plot, a country-by-year heatmap, a group-share area chart, a side-by-side sector comparison, and a world map of victims per million.

### Dataset 3: World Bank country indicators (Badamgarav Battushig)

In [ ]:
# most recent year with good coverage
latest = wb[wb['year'] == 2024]

# top 10 countries by population
top_pop = latest.sort_values('population', ascending=False).head(10)

alt.Chart(top_pop).mark_bar().encode(
    alt.X('population_millions:Q', title='Population (millions)'),
    alt.Y('country:N', sort='-x', title='Country')
).properties(title='Most Populated Countries (2024)', width=450)

In [ ]:
# internet use over time for the main countries in our cyber story
story = ['United States', 'China', 'Russia', 'Iran', 'Ukraine', 'Germany', 'United Kingdom', 'Israel']
story_df = wb[wb['country'].isin(story)]

alt.Chart(story_df).mark_line(point=True).encode(
    alt.X('year:O', title='Year'),
    alt.Y('internet_users_pct:Q', title='Internet users (% of population)', scale=alt.Scale(domain=[0, 100])),
    alt.Color('country:N', title='Country')
).properties(title='Internet Use in Key Cyber Attack Countries', width=500)

In [ ]:
# how connected are countries around the world?
alt.Chart(latest).mark_bar().encode(
    alt.X('internet_users_pct:Q', bin=alt.Bin(step=10), title='Internet users (% of population)'),
    alt.Y('count():Q', title='Number of countries')
).properties(title='Internet Use Across Countries (2024)', width=450)

In [ ]:
# number of countries in each income level
income_order = ['Low', 'Lower middle', 'Upper middle', 'High']

alt.Chart(latest).mark_bar().encode(
    alt.X('income_level:N', sort=income_order, title='Income level'),
    alt.Y('count():Q', title='Number of countries')
).properties(title='Countries by World Bank Income Level', width=350)

In [ ]:
# average internet use by income level
by_income = latest.groupby('income_level')['internet_users_pct'].mean().reset_index()

alt.Chart(by_income).mark_bar().encode(
    alt.X('income_level:N', sort=income_order, title='Income level'),
    alt.Y('internet_users_pct:Q', title='Average internet users (% of population)', scale=alt.Scale(domain=[0, 100]))
).properties(title='Richer Countries Are More Connected (2024)', width=350)

In [ ]:
# do richer countries have more internet users? (scatter plot for EDA only)
alt.Chart(latest).mark_circle(opacity=0.6, size=50).encode(
    alt.X('gdp_per_capita:Q', scale=alt.Scale(type='log'), title='GDP per capita (US$, log scale)'),
    alt.Y('internet_users_pct:Q', title='Internet users (% of population)'),
    alt.Color('income_level:N', sort=income_order, title='Income level')
).properties(title='GDP per Capita vs. Internet Use (2024)', width=450)

#### Summary

What the data shows so far
- The cleaned table has 5,425 rows: 217 countries over 25 years (2000-2024).
- Countries differ hugely in size. In 2024, India had about 1,451 million people and China about 1,409 million, more than four times the United States (about 340 million). Simply counting attacks would make big countries look the most targeted, so we will also look at attacks per million people.
- Internet use grew quickly in all the key countries, from under 50% in 2000 to over 80% by 2024, which matches the rise in cyber incidents after 2010.
- **Richer countries are more connected.** In 2024, about 92% of people in high-income countries used the internet, compared with only 23% in low-income countries.

Questions worth exploring
- Which countries are hit hardest per person once we account for population?
- Do richer countries have more ransomware victims, since ransomware gangs are after money?
- Does internet use or wealth better explain attacks?

Does it need more data?
Yes. This dataset has no cyber attack data on its own. It adds context to EuRepoC and Ransomware.live, and all three can be matched using the 2-letter country code and year.

Visualizations I'm Interested In:
1. A world map of cyber incidents per million people, next to a map of raw counts.
2. A bar chart of ransomware victims per million people by income group.
3. A ranked bar chart of the top 15 countries by attacks per million people.
4. Side-by-side line charts comparing internet growth with yearly cyber incidents.